<a href="https://colab.research.google.com/github/frazeui/Fraud-Detection-AI-Agent/blob/main/fraud_detection_agent_v2_multiagent_project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn nest_asyncio pyngrok

In [2]:
!pip install langchain-groq langgraph-checkpoint-sqlite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.4/163.4 kB 6.0 MB/s eta 0:00:00


In [3]:
import os
from typing import Annotated ,TypedDict
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,SystemMessage,AIMessage
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command,interrupt
import sqlite3
from tenacity import retry,wait_random_exponential,stop_after_attempt


In [4]:
user_profiles={
    "user_101":{"home_country":"UAE","avg_transactions":500},
    "user_102":{"home_country":"Pakistan","avg_transactions":200},
}

In [5]:
import time
@tool
def check_amount_risk(amount:float,user_id:str)->str:
    """verify that either the amount which the users is withdrawing is it average based or fraud """

    time.sleep(2)
    profile=user_profiles.get(user_id)

    print(f"[Debug] finished at {time.time():.2f}")

    if not profile: return "Error: User profile not found "

    avg=profile["avg_transactions"]

    if amount>avg*10:
        return f"High Risk: [{amount}] is {round(amount/avg,1)} higher than user's average"

    elif amount>avg*3:
        return f"Medium Risk: [{amount}] is {round(amount/avg,1)} higher than user's average"

    else:
        return f"Low risk: [{amount}] is wiithin the normal range"

In [6]:
@tool
def check_velocity(transactions_count_last_hour:int)->str:
    """verify the number of transactions last hour either it's a fraud or not """
    time.sleep(2)

    print(f"[Debug] finished at {time.time():.2f}")
    if transactions_count_last_hour>=5:
        return f"High Risk: {transactions_count_last_hour} times which is unusual velocity"

    elif transactions_count_last_hour>=3 and transactions_count_last_hour<5:
        return f"Medium Risk: {transactions_count_last_hour} time which is unusual velocity"

    else:
        return f"Low Risk: {transactions_count_last_hour} times which is normal range"

In [7]:
@tool
def check_location_mismatch(user_id:str,transactions_country:str)->str:
    """Tell about the country that either it's matched or not"""

    time.sleep(2)

    profile=user_profiles.get(user_id)
    if not profile:return "User profile not found"

    home=profile["home_country"]

    print(f"[Debug] finished at {time.time():.2f}")

    if home.lower()!=transactions_country.lower():
        return f"Medium Risk: Transactoin country is [{transactions_country}] and Home country is [{home}]"
    else:
        return f"Low Risk: Transaction country [{transactions_country}] and Home country [{home}] matches "

In [8]:
risk_tools=[check_amount_risk,check_velocity,check_location_mismatch]

In [9]:
class AgentState(TypedDict):
    messages:Annotated[list,add_messages]

In [10]:
from google.colab import userdata

groq_api=userdata.get("groq_api")
llm=ChatGroq(model="llama-3.1-8b-instant",api_key=groq_api)

risk_analyst_llm=llm.bind_tools(risk_tools)
decision_llm=llm

In [11]:
RISK_ANALYST_PROMPT="""
You are a Risk Analyst Agent. Your Only Work is to run risk checks on a transactions using tools - you don't make final decisions or recommendations

Rules:
i-Always run THREE Tools:[check_amount_risk,check_velocity,check_location_mismatch]
ii-After getting all results,summarize each one with factual answer-quote exact tool output don't hallucinate in it
iii-Don't give the final clarification like(SMALL,MEDIUM,HIGH) or (BLOCK/APPROVE),that's the job of Decision Agent not yours
iv-End your summary clearly and send next to another agent which can read your summary easily """


DECISION_AGENT_PROMPT="""
You are Decision Agent,Your Job is not to work with tools & have no access for it  but you'll receive summary result from Risk Analyest Agent and your only job is to:
1.Classify Overall Risk either it's LOW,MEDIUM or HIGH
2.Give the clear recommendations: APPROCE,REVIEW or BLOCK
3.Justify your decision by referencing the specific findings which you were given
4.Never invent new data-only reason the data of Analyest Agent
"""

In [12]:
@retry(wait=wait_random_exponential(min=1,max=20),stop=stop_after_attempt(3))
def risk_analyst_node(state:AgentState):

    messages=state["messages"]
    if not any (isinstance(m,SystemMessage) for m in messages):
        messages=[SystemMessage(content=RISK_ANALYST_PROMPT)]+messages
    response=risk_analyst_llm.invoke(messages)
    return {"messages":[response]}
risk_tool_node=ToolNode(risk_tools)

In [13]:
@retry(wait=wait_random_exponential(min=1,max=20),stop=stop_after_attempt(3))
def decision_agent_node(state:AgentState):
    risk_findings=None
    for m in reversed(state["messages"]):
        if isinstance(m,AIMessage) and not m.tool_calls and m.content:
            risk_findings=m.content
            break
    decision_input=[
        SystemMessage(content=DECISION_AGENT_PROMPT),
        HumanMessage(content=f"Risk Analyst's findings:\n\n{risk_findings}\n\nProvide your final risk classification and recommendation.")
    ]
    response=llm.invoke(decision_input)
    return {"messages":[response]}


In [14]:
def should_continue_risk_analysis(state:AgentState):
    last_message=state["messages"][-1]
    if getattr(last_message,"tool_calls",None):
        return "risk_tools"
    return "decision_agent"

In [15]:
def human_review_node(state:AgentState):
    print("Human Review node reached")
    last_message=state["messages"][-1]
    assessment_text=last_message.content

    needs_review="HIGH" in assessment_text.upper() or 'BLOCK' in assessment_text.upper()

    if not needs_review:return {"messages":[]}
    human_decision = interrupt({
        "question": "Decision Agent flagged this as HIGH risk / BLOCK. Please confirm.",
        "decision_agent_assessment": assessment_text
    })
    confirmation_message=HumanMessage(content=f"[Human Review]: {human_decision}")
    return {"messages":[confirmation_message]}

In [16]:
graph=StateGraph(AgentState)
graph.add_node("risk_analyst",risk_analyst_node)
graph.add_node("risk_tools",risk_tool_node)
graph.add_node("decision_agent",decision_agent_node)
graph.add_node("human_review",human_review_node)

graph.set_entry_point("risk_analyst")

graph.add_conditional_edges("risk_analyst",should_continue_risk_analysis,{"risk_tools":"risk_tools","decision_agent":"decision_agent",END:END})

graph.add_edge("risk_tools","risk_analyst")
graph.add_edge("decision_agent","human_review")
graph.add_edge("human_review",END)

conn=sqlite3.connect("multi_agent_fraud_memory.db",check_same_thread=False)
memory=SqliteSaver(conn)

app=graph.compile(checkpointer=memory)

In [ ]:
print(list(app.get_graph().nodes.keys()))

['__start__', 'risk_analyst', 'risk_tools', 'decision_agent', 'human_review', '__end__']


In [17]:
from fastapi import FastAPI
from pydantic import BaseModel

api=FastAPI(title="Fraud Detection Agent")

class Transactions(BaseModel):
    thread_id:str
    description:str

class HumanDecisionRequest(BaseModel):
    thread_id:str
    decision:str

@api.post("/analyze_transactions")
def analyze_transactions(req:Transactions):
    start=time.time()

    config={"configurable":{"thread_id":req.thread_id}}
    result = app.invoke({
        "messages": [
            HumanMessage(content=req.description)
        ]
    },config=config)


    if "__interrupt__" in result:
        interrupt_data=result["__interrupt__"][0].value
        return {
            "status":"PENDING REQUEST",
            "thread_id":req.thread_id,
            "ai_assessment":interrupt_data["decision_agent_assessment"],
            "message":"High risk detected .Call/human decision with your decision",
            "processing_time_seconds":round(time.time()-start,2)

        }
    return{
        "status": "COMPLETED",
        "thread_id": req.thread_id,
        "final_result": result["messages"][-1].content,
            "processing_time_seconds":round(time.time()-start,2)
    }

@api.post("/human_decision")
def human_decision(req:HumanDecisionRequest):

    config={"configurable":{"thread_id":req.thread_id}}

    result=app.invoke(Command(resume=req.decision),config=config)

    return {
        "status":"COMPLETED",
        "thread_id":req.thread_id,
        "final_result":result["messages"][-1].content
    }


@api.get("/")
def health_check():
    return {"status":"Fraud Detection AI agent API is running...."}

print(f"FastAPI app ready")


FastAPI app ready


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from google.colab import userdata

nest_asyncio.apply()

ngrok_token=userdata.get("ngrok_token")
ngrok.set_auth_token(ngrok_token)

public_url=ngrok.connect(8000)

print(f"Public URL: {public_url}")
print(f"API docs: {public_url}/docs")

config=uvicorn.Config(api,host="0.0.0.0",port=8000,log_level="info")
server=uvicorn.Server(config)
await server.serve()


Public URL: NgrokTunnel: "https://garment-shadily-whimsical.ngrok-free.dev" -> "http://localhost:8000"
API docs: NgrokTunnel: "https://garment-shadily-whimsical.ngrok-free.dev" -> "http://localhost:8000"/docs


INFO:     Started server process [787]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     182.186.25.24:0 - "GET / HTTP/1.1" 200 OK
INFO:     182.186.25.24:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     182.186.25.24:0 - "GET /openapi.json HTTP/1.1" 200 OK
[Debug] Received user_id: 'unknown_user' (type: <class 'str'>)
[Debug] Received user_id: 'unknown_user' (type: <class 'str'>)
[Debug] Received user_id: 'unknown_user' (type: <class 'str'>)
[Debug] Received user_id: 'unknown_user' (type: <class 'str'>)
[Debug] Received user_id: 'user123' (type: <class 'str'>)
[Debug] Received user_id: 'user123' (type: <class 'str'>)
Human Review node reached
INFO:     182.186.25.24:0 - "POST /analyze_transactions HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [787]


# **Agent Evaluation**

In [18]:
import re

TEST_CASES = [
    {
        "id": "TC1_low_normal",
        "description": "Analyze this transaction: user_id=user_101, amount=400, transaction_country=UAE, transaction_count_last_hour=1",
        "expected_risk": "LOW",
    },
    {
        "id": "TC2_high_amount_only",
        "description": "Analyze this transaction: user_id=user_101, amount=15000, transaction_country=UAE, transaction_count_last_hour=1",
        "expected_risk": "HIGH",
    },
    {
        "id": "TC3_high_velocity_only",
        "description": "Analyze this transaction: user_id=user_101, amount=400, transaction_country=UAE, transaction_count_last_hour=7",
        "expected_risk": "HIGH",
    },
    {
        "id": "TC4_location_mismatch_only",
        "description": "Analyze this transaction: user_id=user_101, amount=400, transaction_country=Nigeria, transaction_count_last_hour=1",
        "expected_risk": "MEDIUM",
    },
    {
        "id": "TC5_all_high_signals",
        "description": "Analyze this transaction: user_id=user_101, amount=20000, transaction_country=Nigeria, transaction_count_last_hour=8",
        "expected_risk": "HIGH",
    },
    {
        "id": "TC6_borderline_amount",
        "description": "Analyze this transaction: user_id=user_101, amount=1800, transaction_country=UAE, transaction_count_last_hour=2",
        "expected_risk": "MEDIUM",
    },
    {
        "id": "TC7_normal_user2",
        "description": "Analyze this transaction: user_id=user_102, amount=150, transaction_country=Pakistan, transaction_count_last_hour=1",
        "expected_risk": "LOW",
    },
    {
        "id": "TC8_user2_high_amount",
        "description": "Analyze this transaction: user_id=user_102, amount=3000, transaction_country=Pakistan, transaction_count_last_hour=1",
        "expected_risk": "HIGH",
    },
]


In [55]:
def extract_risk(text:str)->str:
    text=text.upper()
    if re.search(r'\bHIGH\b',text):
        return 'HIGH'
    elif re.search(r'\bMEDIUM\b',text):
        return 'MEDIUM'
    elif re.search(r'\bLOW\b',text):
        return 'LOW'
    return 'UNKOWN'

In [56]:
def run_singal_eval(test_case:dict,thread_id:str)->dict:
    config={"configurable":{"thread_id":thread_id}}

    result=app.invoke({
        "messages":HumanMessage(content=test_case["description"])},config=config)

    if '__interrupt__' in result:
        assessment_text=result['__interrupt__'][0].value["decision_agent_assessment"]

    else:
        assessment_text=result["messages"][-1].content

    predicted_risk=extract_risk(assessment_text)

    return {
        "id":test_case["id"],
        "expected_risk":test_case["expected_risk"],
        "predicted":predicted_risk,
        "correct":predicted_risk==test_case["expected_risk"],
        "raw_response":assessment_text
    }

In [63]:
def run_evaluation():
    results=[]

    for i,test_case in enumerate(TEST_CASES):

        thread_id=f"eval-{test_case['id']}-{i}"
        print(f"Running {test_case['id']}...",end=" ")

        try:
            eval_results=run_singal_eval(test_case,thread_id)
            results.append(eval_results)
            status=("PASS" if eval_results["correct"] else 'FAIL')
            print(f"{status} (expected: {eval_results['expected_risk']},got:{eval_results['predicted']})\n")

        except Exception as e:
            print(f"Error: {e}")
            results.append({
                "id":test_case["id"],
                "expected_risk":test_case['expected_risk'],
                "predicted":'ERROR',
                "correct":False,
                "raw_response":str(e)
                })

    return results

In [64]:
def calculate_results(results:list):
    total=len(results)
    correct=sum(1 for r in results if r["correct"])
    accuracy=(correct/total)*100 if total>0 else 0

    false_negative=[
        r for r in results
        if r["expected_risk"] in ('HIGH','MEDIUM') and r["predicted"] == ('LOW',)
    ]

    false_positives=[
        r for r in results
        if r["expected_risk"] in ('LOW',) and r["predicted"] == ('HIGH','MEDIUM',)
    ]

    print(f"\n{'='*60}")
    print('EVALUATION REPORT')
    print(f"\n{'='*60}")
    print(f"Total Cases: {total}")
    print(f"Correct Cases: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"False negatives: {len(false_negative)}")
    for fn in false_negative:
        print(f"False negative id {fn['id']} expected: {fn['expected_risk']} but got: {fn['predicted']}")

    print(f"False positives: {len(false_positives)}")
    for fn in false_positives:
        print(f"False positive id {fn['id']} expected: {fn['expected_risk']} but got: {fn['predicted']}")

    print(f"\n{'='*60}")

    return {
        "accuracy":accuracy,
        "false_negatives":false_negative,
        "false_positives":false_positives,
        "total":total
    }



In [65]:
if __name__=='__main__':
    results=run_evaluation()
    metrics=calculate_results(results)

Running TC1_low_normal... [Debug] finished at 1786954336.77
[Debug] finished at 1786954336.77
[Debug] finished at 1786954336.77
Human Review node reached
FAIL (expected: LOW,got:HIGH)

Running TC2_high_amount_only... [Debug] finished at 1786954426.14
[Debug] finished at 1786954426.14
[Debug] finished at 1786954426.15
Human Review node reached
PASS (expected: HIGH,got:HIGH)

Running TC3_high_velocity_only... [Debug] finished at 1786954476.51
[Debug] finished at 1786954476.51
[Debug] finished at 1786954476.52
Human Review node reached
PASS (expected: HIGH,got:HIGH)

Running TC4_location_mismatch_only... [Debug] finished at 1786954527.42
[Debug] finished at 1786954527.42
[Debug] finished at 1786954527.42
Human Review node reached
PASS (expected: MEDIUM,got:MEDIUM)

Running TC5_all_high_signals... [Debug] finished at 1786954571.34
[Debug] finished at 1786954571.34
[Debug] finished at 1786954571.34
Human Review node reached
PASS (expected: HIGH,got:HIGH)

Running TC6_borderline_amount... [D